# 3. Download Filtered Scenes

Downloads the scenes kept in `output/my_aoi_scene_search.csv` (produced by
`2_Sentinel_Search.ipynb`). Each sensor has its own dedicated cell below,
so you can run only the ones you need (e.g. skip the huge NISAR RSLC
downloads while still grabbing Sentinel-2). Two different download
mechanisms, depending on where each product actually lives:

- **CMR-sourced** (Sentinel-1 GRD/SLC, NISAR, Sentinel-6, HLS S30/L30) —
  downloaded from their real archive using the `link` column. Requires a
  free NASA Earthdata Login (register at https://urs.earthdata.nasa.gov/
  if you don't have one).
- **Earth Engine-sourced** (Sentinel-2 L2A/L1C, Landsat 8/9 SR, MODIS
  vegetation/snow) — exported directly as a multi-band GeoTIFF clipped to
  your AOI. No login needed.

Sentinel-2 and Landsat are **mixed source**: their EE products (L2A/L1C,
C2 L2) and their CMR/HLS products both appear in the search results, so
those two sensors' cells download from both mechanisms.

**Note on HLS**: each HLS granule is actually many separate per-band files
on the server (B01, B02, ... Fmask, etc). The `link` column stores one
representative band file, not the complete set — ask if you want every
band downloaded instead.

**Note on Sentinel-6**: it's an ocean-only altimetry mission — for a
land-locked AOI, its CMR rows will simply be empty, so there's nothing to
download. That's expected, not a bug.

**Requires**: run `1_AOI_Selection.ipynb` and `2_Sentinel_Search.ipynb`
first, in that order.

📖 Credential issues, or wondering how big a download will be? See
`docs/pdf/03_Download_Guide.pdf`.</cell id="f2708d9b">

## Setup — authenticate and initialize Earth Engine

In [ ]:
import ee

EE_PROJECT = ""  # <-- your GEE-enabled Cloud project

if not EE_PROJECT:
    raise ValueError(
        "Set EE_PROJECT to your own Google Cloud project with the Earth Engine "
        "API enabled — register one free at https://code.earthengine.google.com/register"
    )

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    # auth_mode="localhost" opens your browser and completes automatically via
    # a local redirect — no authorization code to copy/paste.
    ee.Authenticate(auth_mode="localhost")
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialized.")

## Load your AOI and filtered scene list

`AOI_NAME` / `OUTPUT_DIR` must match what you used in the earlier notebooks.

In [ ]:
from pathlib import Path

import pandas as pd

from aoi_export import load_geometry

AOI_NAME = "my_aoi"
OUTPUT_DIR = "output"
DOWNLOAD_DIR = "downloads"

geojson_path = Path(OUTPUT_DIR) / f"{AOI_NAME}.geojson"
csv_path = Path(OUTPUT_DIR) / f"{AOI_NAME}_scene_search.csv"
download_root = Path(DOWNLOAD_DIR)

if not geojson_path.exists():
    raise FileNotFoundError(f"{geojson_path} not found — run 1_AOI_Selection.ipynb first.")
if not csv_path.exists():
    raise FileNotFoundError(f"{csv_path} not found — run 2_Sentinel_Search.ipynb first.")

geometry = load_geometry(geojson_path)
scenes_df = pd.read_csv(csv_path)
print(f"Loaded {len(scenes_df)} scenes from {csv_path}")
scenes_df["product"].value_counts()

## Earthdata Login credentials

Only needed for the CMR-sourced rows (the ones with a `link`). Register a
free account at https://urs.earthdata.nasa.gov/ if you don't have one.

Builds two sessions: `asf_session` (via the official `asf_search` package)
for Sentinel-1 GRD/SLC — its auth handling is purpose-built for ASF and
succeeds where a hand-rolled `requests` session got rejected; and
`earthdata_session` (plain `requests`) for HLS, which already works fine
against LP DAAC.

The password is entered via `getpass` each time you run this cell — it's
never written to disk, so this notebook is safe to commit to git.

In [ ]:
from getpass import getpass

EARTHDATA_USERNAME = ""  # <-- your Earthdata username

if not EARTHDATA_USERNAME:
    raise ValueError(
        "Set EARTHDATA_USERNAME to your own NASA Earthdata Login username — "
        "register free at https://urs.earthdata.nasa.gov/"
    )

EARTHDATA_PASSWORD = getpass("Earthdata password (hidden, not saved to disk): ")

import requests
import asf_search as asf


class _EarthdataSession(requests.Session):
    """NASA's documented Earthdata Login redirect-auth pattern: keeps the
    Authorization header attached across the URS OAuth redirect, but strips
    it once redirected to a third-party host so credentials aren't leaked.
    """

    AUTH_HOST = "urs.earthdata.nasa.gov"

    def __init__(self, username, password):
        super().__init__()
        self.auth = (username, password)

    def rebuild_auth(self, prepared_request, response):
        headers = prepared_request.headers
        url = prepared_request.url
        if "Authorization" in headers:
            original_host = requests.utils.urlparse(response.request.url).hostname
            redirect_host = requests.utils.urlparse(url).hostname
            if (
                original_host != redirect_host
                and redirect_host != self.AUTH_HOST
                and original_host != self.AUTH_HOST
            ):
                del headers["Authorization"]
        return


earthdata_session = _EarthdataSession(EARTHDATA_USERNAME, EARTHDATA_PASSWORD)
asf_session = asf.ASFSession().auth_with_creds(EARTHDATA_USERNAME, EARTHDATA_PASSWORD)
print("Earthdata session ready (requests) and ASF session ready (asf_search).")

## Review what's about to be downloaded

Splits the scene list into CMR-sourced (has a `link`) vs Earth
Engine-sourced (no `link`, downloaded clipped to your small AOI instead —
always small regardless of the original scene size). Set
`CONFIRM_DOWNLOAD = True` below only after checking the CMR total size.

In [ ]:
EE_COLLECTION_BY_PRODUCT = {
    "Sentinel-2 L2A (Surface Reflectance)": "COPERNICUS/S2_SR_HARMONIZED",
    "Sentinel-2 L1C (Top-of-Atmosphere)": "COPERNICUS/S2_HARMONIZED",
    "Landsat 8 C2 L2 (Surface Reflectance)": "LANDSAT/LC08/C02/T1_L2",
    "Landsat 9 C2 L2 (Surface Reflectance)": "LANDSAT/LC09/C02/T1_L2",
    "MODIS Terra Vegetation (NDVI/EVI)": "MODIS/061/MOD13Q1",
    "MODIS Aqua Vegetation (NDVI/EVI)": "MODIS/061/MYD13Q1",
    "MODIS Terra Snow Cover": "MODIS/061/MOD10A1",
    "MODIS Aqua Snow Cover": "MODIS/061/MYD10A1",
}


def _safe_name(product):
    return product.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "-")


cmr_rows = scenes_df[scenes_df["link"].notna()]
ee_rows = scenes_df[scenes_df["product"].isin(EE_COLLECTION_BY_PRODUCT)]

total_cmr_mb = cmr_rows["size_mb"].fillna(0).sum()
print(f"CMR-sourced downloads (needs Earthdata login): {len(cmr_rows)} files, ~{total_cmr_mb:,.0f} MB total")
print(cmr_rows["product"].value_counts().to_string())
print()
print(f"Earth Engine-sourced downloads (clipped to AOI, no login): {len(ee_rows)} files")
print(ee_rows["product"].value_counts().to_string())

CONFIRM_DOWNLOAD = False  # <-- set to True after reviewing the counts/sizes above, then re-run this cell
print(f"\nCONFIRM_DOWNLOAD = {CONFIRM_DOWNLOAD}")
if not CONFIRM_DOWNLOAD:
    print("Set it to True, re-run this cell, then run the two download cells below.")

## Download helpers

Two reusable functions, used by each sensor's cell below — write a
sensor's own cell by filtering `scenes_df` and calling whichever of these
fits its source (some sensors, like Sentinel-2 and Landsat, use both,
since they're partly Earth Engine and partly CMR-sourced).

Routes CMR downloads by the link's actual host, not the product name:
anything hosted by ASF DAAC (`datapool.asf.alaska.edu`,
`*.asf.earthdatacloud.nasa.gov` — covers Sentinel-1 **and** NISAR) goes
through `asf_search` (its auth is what actually works against ASF);
everything else (LP DAAC HLS, PO.DAAC Sentinel-6) uses the plain
`requests` session. Resumable — already-*completed* downloads are
skipped. Downloads write to a temporary `.part` file first, only renamed
to the final filename once fully complete, so a stopped/interrupted
download can never be mistaken for a finished one on a later run.

In [ ]:
from urllib.parse import urlparse

import geemap


def download_cmr_rows(rows):
    """Download a subset of CMR-sourced rows (must each have a non-null 'link')."""
    if not CONFIRM_DOWNLOAD:
        raise RuntimeError("Set CONFIRM_DOWNLOAD = True in the review cell above, then re-run.")

    for _, row in rows.iterrows():
        product_dir = download_root / _safe_name(row["product"])
        product_dir.mkdir(parents=True, exist_ok=True)
        ext = Path(row["link"]).suffix or ".bin"
        out_path = product_dir / f"{row['id']}{ext}"
        tmp_name = out_path.name + ".part"
        tmp_path = product_dir / tmp_name

        if out_path.exists():
            print(f"Skipping (already downloaded): {out_path.name}")
            continue

        size_str = f"{row['size_mb']:.0f} MB" if pd.notna(row.get("size_mb")) else "size unknown"
        print(f"Downloading {row['product']} — {row['id']} ({size_str})...")
        try:
            tmp_path.unlink(missing_ok=True)  # clear any stale partial from an earlier interrupted attempt
            is_asf = "asf" in (urlparse(row["link"]).hostname or "").lower()
            if is_asf:
                asf.download_url(url=row["link"], path=str(product_dir), filename=tmp_name, session=asf_session)
            else:
                with earthdata_session.get(row["link"], stream=True, timeout=300) as r:
                    r.raise_for_status()
                    with open(tmp_path, "wb") as f:
                        for chunk in r.iter_content(chunk_size=1024 * 1024):
                            if chunk:
                                f.write(chunk)
            tmp_path.rename(out_path)
            print(f"  -> saved to {out_path}")
        except BaseException:
            tmp_path.unlink(missing_ok=True)
            raise

    print(f"Done: {len(rows)} row(s) processed.")


def download_ee_rows(rows):
    """Download a subset of Earth Engine-sourced rows, clipped to the AOI."""
    if not CONFIRM_DOWNLOAD:
        raise RuntimeError("Set CONFIRM_DOWNLOAD = True in the review cell above, then re-run.")

    for _, row in rows.iterrows():
        collection_id = EE_COLLECTION_BY_PRODUCT[row["product"]]
        product_dir = download_root / _safe_name(row["product"])
        product_dir.mkdir(parents=True, exist_ok=True)
        out_path = product_dir / f"{row['id']}.tif"

        if out_path.exists():
            print(f"Skipping (already downloaded): {out_path.name}")
            continue

        print(f"Downloading {row['product']} — {row['id']}...")
        img = ee.Image(f"{collection_id}/{row['id']}").clip(geometry)
        geemap.ee_export_image(
            img, filename=str(out_path), scale=row["resolution_m"], region=geometry, file_per_band=False
        )

    print(f"Done: {len(rows)} row(s) processed.")

## Download Sentinel-1

CMR-sourced (ASF DAAC). GRD/SLC files are large (~1–4.5 GB each).

In [ ]:
sentinel1_rows = scenes_df[scenes_df["sensor"] == "Sentinel-1"]
download_cmr_rows(sentinel1_rows)

## Download Sentinel-2

Mixed source: L2A/L1C come from Earth Engine (clipped to your AOI, small);
HLS S30 is CMR-sourced (LP DAAC), full per-band files.

In [ ]:
sentinel2_rows = scenes_df[scenes_df["sensor"] == "Sentinel-2"]
s2_ee_rows = sentinel2_rows[sentinel2_rows["product"].isin(EE_COLLECTION_BY_PRODUCT)]
s2_cmr_rows = sentinel2_rows[sentinel2_rows["link"].notna()]

download_ee_rows(s2_ee_rows)
download_cmr_rows(s2_cmr_rows)

## Download Landsat

Mixed source: C2 L2 Surface Reflectance comes from Earth Engine (clipped
to your AOI, small); HLS L30 is CMR-sourced (LP DAAC), full per-band file.

In [ ]:
landsat_rows = scenes_df[scenes_df["sensor"] == "Landsat"]
landsat_ee_rows = landsat_rows[landsat_rows["product"].isin(EE_COLLECTION_BY_PRODUCT)]
landsat_cmr_rows = landsat_rows[landsat_rows["link"].notna()]

download_ee_rows(landsat_ee_rows)
download_cmr_rows(landsat_cmr_rows)

## Download NISAR

CMR-sourced (ASF DAAC). **RSLC files are huge (~25 GB each)** — GCOV and
Soil Moisture are far smaller. Consider downloading a subset of
`nisar_rows` instead of the full set if you don't need every product.

In [ ]:
nisar_rows = scenes_df[scenes_df["sensor"] == "NISAR"]
download_cmr_rows(nisar_rows)

## Download MODIS

Earth Engine-sourced (Terra/Aqua Vegetation + Snow Cover). Clipped to
your AOI, no login needed.

In [ ]:
modis_rows = scenes_df[scenes_df["sensor"] == "MODIS"]
download_ee_rows(modis_rows)

## Download Sentinel-6

CMR-sourced (PO.DAAC). Ocean-only altimetry mission — for a land-locked
AOI this will typically be empty, which is expected, not a bug.

In [ ]:
sentinel6_rows = scenes_df[scenes_df["sensor"] == "Sentinel-6"]
download_cmr_rows(sentinel6_rows)

## Summary

In [ ]:
downloaded = list(download_root.rglob("*.*")) if download_root.exists() else []
total_bytes = sum(f.stat().st_size for f in downloaded)
print(f"{len(downloaded)} file(s) in {download_root.resolve()}, {total_bytes / (1024 ** 2):,.1f} MB total")
for f in sorted(downloaded):
    print(f" - {f.relative_to(download_root)} ({f.stat().st_size / (1024 ** 2):.1f} MB)")